# Speech Recognition and Speech-to-Speech Translation

<a href="https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/02_pipelines_exercise.ipynb" target="_blank">
  <img src="https://raw.githubusercontent.com/HassanAlgoz/dl/main/assets/Open%20in%20Colab-F9AB00.svg" alt="Open in Colab" height="50"/>
</a>

Whisper turns speech into text. By chaining ASR, translation, and text-to-speech pipelines, you can build a **speech-to-speech translation (STST)** system — audio in one language, spoken audio out in another.


**Goal:** Transcribe your own English and Arabic recordings with timestamps, then chain ASR → translation → TTS into a speech-to-speech pipeline.

**Install:** `%pip install -qqq transformers datasets accelerate torch sentencepiece`

**Model:** [openai/whisper-large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) (0.8B params, F16)


In [ ]:
# --- Setup: Clone repo & cd into correct folder (Colab only) ---
import os
import sys
import subprocess

if "google.colab" in sys.modules:
    repo_url = "https://github.com/HassanAlgoz/dl.git"
    lab_folder = "dl/modules/pipelines/02-pipelines"

    # Only clone if the folder doesn't exist
    if not os.path.exists(lab_folder):
        subprocess.run(["git", "clone", repo_url])

    # Change working directory to the lab folder
    os.chdir(lab_folder)


## Imports


In [ ]:
# %pip install -qqq transformers datasets accelerate torch sentencepiece

In [ ]:
import sys

import IPython.display as ipd

import torch
from transformers import pipeline

## Exercise 1: Speech Recognition

We'll use [whisper-large-v3-turbo](https://huggingface.co/openai/whisper-large-v3-turbo) which has **0.8B params** and is **F16**, so it is small.

For a 1 minute audio:

- on CPU takes about 3 minutes
- on GPU takes about 3 seconds

### Your tasks

1. Open the model card [**Usage** section](https://huggingface.co/openai/whisper-large-v3-turbo#usage) and copy/adapt the code so you know how to use the model properly.
2. Record or obtain **two short audio clips** (~15–60 s): one in **English**, one in **Arabic**. Upload them yourself (phone memo, Colab recorder, etc.).
3. Transcribe **each** file. Your output should include **when** things were spoken — check the model card for how to do that.
4. For the Arabic clip, the source language is known ahead of time — see what the model card recommends.


In [ ]:
! pip install --upgrade pip
! pip install --upgrade transformers datasets[audio] accelerate

### Upload your audio

Run the cell below to upload files in Colab, or set the path strings directly if running locally.


In [ ]:
from pathlib import Path

english_audio = Path("/content/manim_explained.mp4")
english_audio.exists()

### Build the ASR pipeline

Copy the setup code from the [model card Usage section](https://huggingface.co/openai/whisper-large-v3-turbo#usage).


In [ ]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)

### Transcribe English

Listen to your clip, then transcribe it.


In [ ]:
ipd.display(ipd.Audio(english_audio))

In [ ]:
result = pipe(str(english_audio), return_timestamps=True)
result

### To Transcribe Arabic Speech

```python
result = pipe(
    sample,
    generate_kwargs={
        "language": "arabic"
      }
)
```

## Exercise 2: Joining Pipelines

![](https://github.com/HassanAlgoz/dl/blob/main/modules/pipelines/assets/joining_pipelines.png?raw=1)

Here's an [interactive demo of speech-to-speech translation](https://course-demos-speech-to-speech-translation.hf.space).

### Scenario

Build a **speech-to-speech translation (STST)** chain on one of your Exercise 1 recordings (or a new clip). You choose the source → target language; find matching models on the Hub.

### Required steps

1. **ASR** — reuse your Whisper pipeline from Exercise 1
2. **Translation** — `pipeline("translation", model=...)`; browse [Helsinki-NLP opus-mt models](https://huggingface.co/models?sort=trending&search=Helsinki-NLP) for `{src}-en` or `{src}-{tgt}` pairs
3. **TTS** — load a text-to-speech model for your **target** language (see [tasks_audio.ipynb](https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/tasks_audio.ipynb) for an Arabic VITS example; pick any suitable model card for other languages)
4. **Chain** — wire the three steps so audio in produces spoken audio out; display the final waveform with `IPython.display.Audio`

**Hints:**
- Pass the ASR `text` field into the translator; pass `translation_text` into TTS.
- Reuse patterns from [05_seq2seq.ipynb](https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/05_seq2seq.ipynb) (translation) and [tasks_audio.ipynb](https://colab.research.google.com/github/HassanAlgoz/dl/blob/main/modules/pipelines/02-pipelines/tasks_audio.ipynb) (TTS).
- If the source language is already your TTS target language, translate to a **different** language so you demonstrate a full STST pipeline.


### Step 2 — Translation pipeline


In [ ]:
full_transcript = result['text']
type(full_transcript)

### Step 3 — Text-to-Speech model


In [ ]:
# Correct installation for NeMo and dependencies
!pip install git+https://github.com/NVIDIA/NeMo.git@main#egg=nemo_toolkit[tts]
!pip install kaldialign

In [ ]:
from nemo.collections.tts.models import MagpieTTSModel
import IPython.display as ipd
import torch

# Load the model
# Note: This might take a few minutes to download the weights
tts_model = MagpieTTSModel.from_pretrained("nvidia/magpie_tts_multilingual_357m")
tts_model.to("cuda" if torch.cuda.is_available() else "cpu")
tts_model.eval()

# Setup parameters
transcript = "Hello world from NeMo Text to Speech."
language = "en"
speaker_map = {"Aria": 0, "Jason": 1, "John": 2, "Leo": 3, "Sofia": 4}
speaker_idx = speaker_map["Sofia"]

# Generate audio
with torch.no_grad():
    audio, audio_len = tts_model.do_tts(transcript, language=language, speaker_index=speaker_idx)

# Display the result
# audio is usually a torch tensor, we convert to numpy for display
ipd.display(ipd.Audio(audio.cpu().numpy(), rate=44100))

### Step 4 — Chain into `speech_to_speech(audio_path)`

Write a function that takes an audio file path and returns playable output audio. Run it on one of your recordings.


In [ ]:
def speech_to_speech(audio_path, target_lang='en', speaker='Sofia'):
    # 1. ASR (Speech to Text)
    print(f'Transcribing {audio_path}...')
    asr_result = pipe(audio_path)
    source_text = asr_result['text']
    print(f'Source Text: {source_text}')

    # 2. Translation (Placeholder: keeps same language or can be extended)
    translated_text = source_text

    # 3. TTS (Text to Speech)
    print(f'Generating speech for: {translated_text}')
    speaker_idx = speaker_map.get(speaker, 4)

    with torch.no_grad():
        audio, _ = tts_model.do_tts(
            translated_text,
            language=target_lang,
            speaker_index=speaker_idx
        )

    # Return audio for playback
    return ipd.Audio(audio.cpu().numpy(), rate=44100)

# Set path and run
english_audio = "/content/manim_explained.mp4"
speech_to_speech(english_audio)

## Conclusion

- Model cards are the source of truth for how to load and call a checkpoint.
- Timestamps let you align transcription with the original audio.
- Real-world speech applications often chain single-task pipelines — ASR, translation, and TTS — into compound systems like STST.
